# Assignment 01 - Vietnam House Price Prediction Intelligent System

Dataset: House Price Prediction Dataset Vietnam - 2024  
Task: supervised regression to estimate house listing price in billion VND.

This notebook is the scientific workflow for the House Price part of Assignment 01. It uses the revised active representation with exactly 11 model features.

## Revised Feature Strategy

Raw `Address` is not used directly as an ML feature because it is high-cardinality and contains detailed text. The pipeline parses:

- `Province` from the last comma-separated segment.
- `District` from the second-last comma-separated segment.
- `Location = District + ", " + Province`.

The model uses `Location` as one categorical feature. `Province` and `District` may still appear in the Knowledge Graph as geographic context, but they are not separate ML features.

## Active 11 Model Features

Numeric features:

1. Area
2. Frontage
3. Access Road
4. Floors
5. Bedrooms
6. Bathrooms

Categorical features:

7. House direction
8. Balcony direction
9. Legal status
10. Furniture state
11. Location

In [1]:
from pathlib import Path
import json
import sys

import joblib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)

/home/hiubeo/Documents/TKHTTM/Assignment01/Assignment01_HousePrice


In [2]:
from house_price_pipeline import HOUSE_FEATURES_10, HOUSE_FEATURES_11, REPRESENTATION_VERSION, run_pipeline

print(REPRESENTATION_VERSION)
print(HOUSE_FEATURES_11)
assert len(HOUSE_FEATURES_11) == 11
assert HOUSE_FEATURES_11[-1] == "Location"
assert len(HOUSE_FEATURES_10) == 10
assert "Location" not in HOUSE_FEATURES_10

house_11_feature_v2
['Area', 'Frontage', 'Access Road', 'House direction', 'Balcony direction', 'Floors', 'Bedrooms', 'Bathrooms', 'Legal status', 'Furniture state', 'Location']


## Execute Scientific Pipeline

The pipeline performs data loading, deterministic address parsing, preprocessing, train/test split, cross-validation experiments, final held-out testing, figure export, and compressed model artifact export. Streamlit does not retrain models at runtime.

In [3]:
summary = run_pipeline(PROJECT_ROOT, verbose=True)
summary.keys()

Experiment 1 done: Linear Regression {'MAE': 1.2902631992278923, 'MSE': 2.732970375137627, 'RMSE': 1.6529974263688785, 'R2': 0.44187174139651636, 'MAPE': 0.27711148397718627}


Experiment 1 done: Decision Tree {'MAE': 1.3247596554560779, 'MSE': 3.520240518416994, 'RMSE': 1.876019956628069, 'R2': 0.2811035909258807, 'MAPE': 0.2729095911668774}


Experiment 1 done: Random Forest {'MAE': 1.0569265924399793, 'MSE': 2.0477374356404257, 'RMSE': 1.430669738199871, 'R2': 0.5818011148049183, 'MAPE': 0.22624310142240373}


Experiment 1 done: Extra Trees {'MAE': 1.0662233818071847, 'MSE': 2.1478443085624237, 'RMSE': 1.4651234159794804, 'R2': 0.5613823985543288, 'MAPE': 0.22564230663449472}


Experiment 1 done: Gradient Boosting {'MAE': 1.318503367220559, 'MSE': 2.702899021865874, 'RMSE': 1.6439503759556184, 'R2': 0.4479926503835315, 'MAPE': 0.29856071984032073}


Experiment 2 done: max_depth 5 {'MAE': 1.477347828491642, 'MSE': 3.3666433105971287, 'RMSE': 1.834774664283848, 'R2': 0.31238912130124624, 'MAPE': 0.33781383608591203}


Experiment 2 done: max_depth 10 {'MAE': 1.3241463069665182, 'MSE': 2.8222569084973004, 'RMSE': 1.6798316773244522, 'R2': 0.4235685738545034, 'MAPE': 0.2955740490830778}


Experiment 2 done: max_depth 20 {'MAE': 1.164624991165422, 'MSE': 2.308339606557218, 'RMSE': 1.5190676856968832, 'R2': 0.5285824166795137, 'MAPE': 0.2538092904937832}


Experiment 2 done: max_depth None {'MAE': 1.0569265924399793, 'MSE': 2.047737435640426, 'RMSE': 1.430669738199871, 'R2': 0.5818011148049183, 'MAPE': 0.22624310142240373}


dict_keys(['dataset', 'eda_charts', 'features', 'split', 'memory_check', 'baseline', 'experiment1', 'experiment2', 'candidate', 'experiment3', 'final_test', 'demos', 'artifact_sizes_bytes'])

## Experiment 1 - Five Regressors with 11 Features

All five regressors use the same 11-feature representation, preprocessing protocol, train split, 5-fold CV, target, and metrics. The changed variable is the regression algorithm.

In [4]:
pd.DataFrame(summary["experiment1"]["compact_table"]).sort_values("RMSE")

,Model,MAE,MSE,RMSE,R2,MAPE
2,Random Forest,1.056927,2.047737,1.430670,0.581801,0.226243
3,Extra Trees,1.066223,2.147844,1.465123,0.561382,0.225642
4,Gradient Boosting,1.318503,2.702899,1.643950,0.447993,0.298561
0,Linear Regression,1.290263,2.732970,1.652997,0.441872,0.277111
1,Decision Tree,1.324760,3.520241,1.876020,0.281104,0.272910


## Experiment 2 - Random Forest max_depth

This controlled experiment keeps the 11 features fixed and changes only `RandomForestRegressor(max_depth)` among 5, 10, 20, and None.

In [5]:
pd.DataFrame(summary["experiment2"]["table"]).sort_values("RMSE")

,max_depth,MAE,MSE,RMSE,R2,MAPE
3,None,1.056927,2.047737,1.430670,0.581801,0.226243
2,20,1.164625,2.308340,1.519068,0.528582,0.253809
1,10,1.324146,2.822257,1.679832,0.423569,0.295574
0,5,1.477348,3.366643,1.834775,0.312389,0.337814


## Experiment 3 - Location Ablation

This revised experiment compares exactly 10 features without `Location` against 11 features with `Location`. The changed variable is the inclusion of the combined geographic model feature.

In [6]:
pd.DataFrame(summary["experiment3"]["table"]).sort_values("RMSE")

,Representation,Feature Count,MAE,MSE,RMSE,R2,MAPE
2,Difference (11 - 10),1,-0.390808,-1.411814,-0.429256,0.288392,-0.098413
1,11 With Location,11,1.056927,2.047737,1.430670,0.581801,0.226243
0,10 Without Location,10,1.447735,3.459552,1.859925,0.293409,0.324656


## Final Model and Held-out Test

The final scientific model is selected by training cross-validation RMSE, then evaluated once on the held-out test set.

In [7]:
registry = json.loads((PROJECT_ROOT / "models" / "model_registry.json").read_text(encoding="utf-8"))
metadata = json.loads((PROJECT_ROOT / "models" / "ui_metadata.json").read_text(encoding="utf-8"))

print("Final model:", registry["scientific_final_model"])
print("Feature count:", registry["feature_count"])
print("Representation:", registry["representation_version"])
pd.DataFrame([registry["final_test_metrics"]])

Final model: Random Forest
Feature count: 11
Representation: house_11_feature_v2


,model,MAE,MSE,RMSE,R2,MAPE
0,Random Forest,1.043087,2.037064,1.427257,0.582219,0.221385


## Artifact Sanity Check

All deployment models are saved with xz compression and can predict from a DataFrame containing exactly the 11 active columns.

In [8]:
sample = pd.DataFrame([{feature: metadata["numeric_ranges"][feature]["median"] if feature in metadata["numeric_ranges"] else metadata["location_values"][0] if feature == "Location" else np.nan for feature in registry["selected_features"]}], columns=registry["selected_features"])

predictions = []
for model_name, model_info in registry["models"].items():
    model = joblib.load(PROJECT_ROOT / "models" / model_info["file"])
    predictions.append({"model": model_name, "prediction_billion_vnd": float(model.predict(sample)[0])})

pd.DataFrame(predictions)

,model,prediction_billion_vnd
0,Linear Regression,2.736765
1,Decision Tree,2.900000
2,Random Forest,2.686300
3,Extra Trees,2.670500
4,Gradient Boosting,4.534802


## Conclusion

The active House Price system now uses exactly 11 model features. `Location` is empirically tested through the 10-vs-11 ablation and remains the single ML location feature. District and Province are reserved for Knowledge Graph context rather than separate model inputs. Predictions are educational estimates, not official valuations.